In [ ]:
import json
import math
from typing import Dict, Any
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

def get_cosine_similarity(proposal_text: str, project_desc: str) -> float:
    """Calculates semantic distance between the proposal and project description."""
    if not proposal_text or not project_desc:
        return 0.0
        
    doc = Document(page_content=proposal_text)
    vectorstore = FAISS.from_documents([doc], embeddings)
    results = vectorstore.similarity_search_with_relevance_scores(project_desc, k=1)
    
    if results:
        score = results[0][1]
        return max(0.0, min(1.0, float(score)))
    return 0.0
def get_llm_metrics(proposal: Dict[str, Any], project: Dict[str, Any]) -> Dict[str, float]:
    """Forces the LLM to output exact floats for relevance and clarity, now factoring in milestones."""

    milestones = proposal.get("proposed_milestones", [])
    milestone_text = "None provided."
    if milestones:
        milestone_text = "\n".join(
            [f"- {m.get('title', 'Task')}: ${m.get('amount', 0)} (Due: {m.get('due_date', 'N/A')})" for m in milestones]
        )

    prompt_template = PromptTemplate.from_template("""
    You are evaluating a freelancer's proposal against a client's project.
    
    Project Title: {title}
    Project Description: {desc}
    
    Freelancer Proposal (Cover Letter):
    {proposal}
    
    Proposed Milestones:
    {milestones}
    
    Evaluate two metrics from 0.0 (terrible) to 1.0 (perfect):
    1. 'llm_relevance': How accurately does the proposal address the specific project needs?
    2. 'clarity': How clear, professional, and well-structured is the proposal? (Heavily weigh the presence and logic of the Proposed Milestones here. Good milestones = high clarity).
    
    Respond STRICTLY in JSON format: {{"llm_relevance": 0.85, "clarity": 0.90}}
    """)
    
    prompt = prompt_template.format(
        title=project.get("title", ""),
        desc=project.get("description", ""),
        proposal=proposal.get("cover_letter", ""),
        milestones=milestone_text
    )
    
    try:
        response = llm.invoke(prompt)
        clean_json = response.content.replace("```json", "").replace("```", "").strip()
        data = json.loads(clean_json)
        
        return {
            "llm_relevance": float(data.get("llm_relevance", 0.0)),
            "clarity": float(data.get("clarity", 0.0))
        }
    except Exception as e:
        print(f"LLM Parsing Error: {e}")
        return {"llm_relevance": 0.0, "clarity": 0.0}

def evaluate_proposal(proposal: Dict[str, Any], project: Dict[str, Any]) -> Dict[str, Any]:

    bid_amount = float(proposal.get("bid_amount", 0.0))
    max_budget = float(project.get("budget_range", {}).get("max", 0.0))
    
    cosine_score = get_cosine_similarity(proposal.get("cover_letter", ""), project.get("description", ""))
    llm_data = get_llm_metrics(proposal, project)

    relevance = (0.6 * cosine_score) + (0.4 * llm_data["llm_relevance"])

    quality = (0.7 * relevance) + (0.3 * llm_data["clarity"])
    
    overshoot_ratio = 2.0
    
    if bid_amount > max_budget and max_budget > 0:
        penalty_factor = overshoot_ratio * ((bid_amount - max_budget) / max_budget)
        price_penalty = min(1.0, penalty_factor)
    else:
        price_penalty = 0.0
    final_score = quality * (1.0 - price_penalty)
    low_bid = float(project.get("budget_range", {}).get("min", 0.0))
    if max_budget > low_bid:
        normalised_bid = (bid_amount - low_bid) / (max_budget - low_bid)
        normalised_bid = max(0.0, min(1.0, normalised_bid))
    else:
        normalised_bid = 0.0
        
    return {
        "proposal_id": proposal.get("proposal_id"),
        "final_score": round(final_score, 4),
        "breakdown": {
            "relevance": round(relevance, 4),
            "clarity":round(llm_data["clarity"],4),
            "quality": round(quality, 4),
            "price_penalty": round(price_penalty, 4),
            "normalised_bid": round(normalised_bid, 4)
        }
    }

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3535.64it/s]


In [13]:
with open('projects.json','r') as file:
    projects=json.load(file)

In [14]:
with open('proposals.json','r') as file:
    proposals=json.load(file)

In [15]:
for p in proposals:
    print(evaluate_proposal(p,projects[0]))

{'proposal_id': 'prop_1a2b3c4d', 'final_score': 0.8531, 'breakdown': {'relevance': 0.8115, 'clarity': 0.95, 'quality': 0.8531, 'price_penalty': 0.0, 'normalised_bid': 0.8667}}
{'proposal_id': 'prop_5e6f7g8h', 'final_score': 0.0884, 'breakdown': {'relevance': 0.1048, 'clarity': 0.05, 'quality': 0.0884, 'price_penalty': 0.0, 'normalised_bid': 0.0}}
{'proposal_id': 'prop_9i0j1k2l', 'final_score': 0.246, 'breakdown': {'relevance': 0.3674, 'clarity': 0.9, 'quality': 0.5272, 'price_penalty': 0.5333, 'normalised_bid': 1.0}}


C:\Users\M AVINASH REDDY\AppData\Local\Temp\ipykernel_9348\3226532230.py:26: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='ef4b8f88-8517-4f3b-9dee-60746d783eb1', metadata={}, page_content='I specialize in Agentic AI and RAG architectures. I have built highly scalable dual-encoder retrieval setups using LangChain, HuggingFace embeddings, and FAISS. For your unstructured PDFs, I will implement semantic chunking with overlap to preserve context boundaries before passing the context windows to the LLM. I am ready to start testing prototypes this week.'), np.float32(-0.30009496))]
  results = vectorstore.similarity_search_with_relevance_scores(project_desc, k=1)


{'proposal_id': 'prop_3m4n5o6p', 'final_score': 0.03, 'breakdown': {'relevance': 0.0, 'clarity': 0.1, 'quality': 0.03, 'price_penalty': 0.0, 'normalised_bid': 0.0}}


C:\Users\M AVINASH REDDY\AppData\Local\Temp\ipykernel_9348\3226532230.py:26: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='f1f0691b-eba6-462f-8eeb-00048306d431', metadata={}, page_content="I have extensive experience in Python and Natural Language Processing. While I haven't explicitly built a dual-encoder RAG from scratch, I have built several sentiment analysis APIs using FastAPI and Transformers. I am a very fast learner and my hourly rate is at the lowest end of your budget to reflect the learning curve I will undertake to build this pipeline for you."), np.float32(-0.3805344))]
  results = vectorstore.similarity_search_with_relevance_scores(project_desc, k=1)


{'proposal_id': 'prop_7q8r9s0t', 'final_score': 0.059, 'breakdown': {'relevance': 0.02, 'clarity': 0.15, 'quality': 0.059, 'price_penalty': 0.0, 'normalised_bid': 0.0}}


In [ ]:
projects[0]

{'project_id': 'proj_a1b2c3d4',
 'client_id': 'client_98765',
 'title': 'Database Normalization and PostgreSQL Migration',
 'description': 'We need a database architect to redesign our legacy inventory schema. The current structure has multiple BCNF violations causing update anomalies. You will be responsible for normalizing the tables to 3NF/BCNF while ensuring dependency preservation, and migrating the existing data to a new PostgreSQL instance.',
 'required_skills': ['PostgreSQL',
  'Database Normalization',
  'Data Migration',
  'SQL'],
 'budget_range': {'min': 1500, 'max': 3000},
 'deadline': '2026-06-30T23:59:59Z',
 'project_type': 'Fixed',
 'status': 'Open',
 'created_at': '2026-05-20T15:17:31Z'}

In [18]:
proposals[2]

{'proposal_id': 'prop_9i0j1k2l',
 'project_id': 'proj_a1b2c3d4',
 'freelancer_id': '9a8b7c6d-5e4f-3a2b-1c0d-9e8f7a6b5c4d',
 'bid_amount': 3800.0,
 'estimated_days': 30,
 'cover_letter': 'Migrating to PostgreSQL is only half the battle. To truly fix your architecture, you also need a Redis caching layer and a C++ microservice to handle the transaction loads. I have quoted to rebuild your backend entirely. This exceeds your initial budget, but it is the correct engineering approach for long-term scalability.',
 'proposed_milestones': [{'milestone_id': 'mile_301',
   'title': 'Architecture Review & Redis Setup',
   'amount': 1200.0,
   'due_date': '2026-06-10T00:00:00Z'},
  {'milestone_id': 'mile_302',
   'title': 'Schema Normalization',
   'amount': 1000.0,
   'due_date': '2026-06-20T00:00:00Z'},
  {'milestone_id': 'mile_303',
   'title': 'C++ Microservice Integration',
   'amount': 1600.0,
   'due_date': '2026-06-30T00:00:00Z'}],
 'attachments': [],
 'status': 'Pending',
 'created_at': 